# Lesson 2: Test functions and test files

This notebook goes with [Lesson 2 of *Unit Testing in Python*](https://acatlin.github.io/python-unit-testing/lessons/0002-test-functions-and-test-files.html). The explanations are on that page. The code is here, ready to run.

- Run the cells from top to bottom, one at a time: click a cell and press Shift+Enter.
- This is your own copy. Change anything you like; nothing you do here alters the course. To keep your work, choose File, then Save a copy in Drive.

## Set up

Colab comes with older versions of pandas and pytest than the lesson page was written with. The next cell installs the versions the page uses, so that what you see here matches it. It can take a minute.

pip may end with a message that other packages in Colab want the older pandas. This notebook does not use them, so carry on. If Colab asks you to restart the session, do that, then continue from the cell after this one.

In [ ]:
%pip install -q "pandas==3.1.1" "pytest==9.2.0"

Check what you have. A line that starts with `!` goes to the computer's command line, not to Python: it is this notebook's stand-in for the terminal.

In [ ]:
!python --version
!python -m pytest --version
!python -c "import pandas; print('pandas', pandas.__version__)"

## How files and commands work here

The lesson has you save files in an editor and type commands in a terminal. A notebook has neither, so two stand-ins do the job.

- **`%%writefile name`** on the first line of a cell saves the rest of the cell as a file, in place of running it. To change the file, edit the cell and run it again. (You can also open the file from the folder icon at the left edge of Colab.)
- **`!`** at the start of a line sends it to the command line. Every `!` line starts in the notebook's own folder, so the commands below begin with `cd cleaning-project &&` to step into the project first.

**What will look different from the lesson page.** The page shows output captured on Windows with Python 3.14. Colab runs Linux with its own Python. So the `platform` line says `linux` and another Python version, `rootdir:` is a folder under `/content`, paths are written with `/` in place of `\`, the rules of `=` signs may be wider or narrower, and the timings differ. The results are the same: the dots and `F`s, the lines marked `E`, and the counts on the last line.

## Give tests a home

Make the project folder with a `tests/` folder inside it, then save the two files: `clean.py` with the functions, and `tests/test_clean.py` with the tests.

In [ ]:
import os

os.makedirs("cleaning-project/tests", exist_ok=True)

In [ ]:
%%writefile cleaning-project/clean.py
"""Data-cleaning helpers: the starting point for Lesson 2.

Each function is small on purpose: small functions are easy to test.
"""

import re
import warnings

import pandas as pd


def rescale01(x):
    """Rescale a numeric column so its smallest value is 0 and its largest is 1."""
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo)


def clean_names(names):
    """Turn messy column names into lower-case snake_case."""
    cleaned = []
    for name in names:
        name = re.sub(r"[^a-z0-9]+", "_", name.lower())
        cleaned.append(name.strip("_"))
    return cleaned


def parse_money(x):
    """Convert a column of strings such as "$1,200.50" to numbers."""
    digits = x.str.replace(r"[$,]", "", regex=True)
    numbers = pd.to_numeric(digits, errors="coerce").astype("float64")
    n_bad = int(numbers.isna().sum() - x.isna().sum())
    if n_bad > 0:
        warnings.warn(f"{n_bad} value(s) could not be parsed as money", stacklevel=2)
    return numbers


def summarise_by(data, group, value):
    """Count rows and average one column within each group."""
    return data.groupby(group, as_index=False).agg(
        n=(value, "size"),
        mean=(value, "mean"),
    )

In [ ]:
%%writefile cleaning-project/tests/test_clean.py
import pandas as pd

from clean import clean_names, parse_money, rescale01


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip_code"]


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]

## Run everything

One command, from the project folder. You should see `3 passed`.

In [ ]:
!cd cleaning-project && python -m pytest

Add `-v` and pytest lists every test by name.

In [ ]:
!cd cleaning-project && python -m pytest -v

## Reading a failure

Suppose you had typed the expected name as `"zip-code"`. The next cell saves the test file with that mistake in it, and the one after runs the tests. Read the report in three steps: **what** (the name under `FAILURES`), **where** (the line marked `>`), **why** (the lines marked `E`).

In [ ]:
%%writefile cleaning-project/tests/test_clean.py
import pandas as pd

from clean import clean_names, parse_money, rescale01


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip-code"]


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]

In [ ]:
!cd cleaning-project && python -m pytest

The function was fine and the test was mistyped. Put the test file right and run again.

In [ ]:
%%writefile cleaning-project/tests/test_clean.py
import pandas as pd

from clean import clean_names, parse_money, rescale01


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip_code"]


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]

In [ ]:
!cd cleaning-project && python -m pytest

## The common trap

Plain `pytest`, without `python -m`, cannot find `clean.py`. The run stops with `ModuleNotFoundError: No module named 'clean'`. Nothing is wrong with the tests.

In [ ]:
!cd cleaning-project && pytest

## Try it

Steps 1 and 2 of the lesson are done: you have the project and have seen `3 passed`.

3. In the cell that saves `tests/test_clean.py` (the one just above "The common trap"), add a third assertion to the `parse_money()` test, for the input `"$3,000"`. Run that cell, then the cell below. The count is still 3, because pytest counts tests. Now move that assertion into a new test of its own, with a name that says what it checks, and watch the count become 4.

In [ ]:
!cd cleaning-project && python -m pytest

4. Rename the test file to `clean_tests.py` and run again. You should see `collected 0 items` and `no tests ran`. The third cell renames it back.

In [ ]:
os.rename("cleaning-project/tests/test_clean.py", "cleaning-project/tests/clean_tests.py")

In [ ]:
!cd cleaning-project && python -m pytest

In [ ]:
os.rename("cleaning-project/tests/clean_tests.py", "cleaning-project/tests/test_clean.py")

In [ ]:
!cd cleaning-project && python -m pytest

Now go back to the [lesson page](https://acatlin.github.io/python-unit-testing/lessons/0002-test-functions-and-test-files.html) for the quiz. The Lesson 3 notebook starts from this project with the fourth test already added.